# Sistemas de Preguntas y Respuestas con Transformers (**SOLVED**)

**Task:** Question Answering (QA) sobre contexto dado + mini QA "open-domain" usando un modelo preentrenado de `transformers`.

**Modelo utilizado:** `deepset/roberta-base-squad2` (inglés, extractive QA).

---

Esta es la **versión resuelta** del notebook. Incluye todas las celdas de código completas y comentadas.
Se recomienda intentar primero la versión **STUDENT** y usar esta como referencia.

---

## Objetivos del notebook

En este notebook vas a:

1. Cargar un modelo de **QA extractivo** preentrenado usando `transformers`.
2. Probar el modelo en ejemplos sencillos de "pregunta + contexto" (tipo SQuAD).
3. Implementar una mini-evaluación con **Exact Match (EM)** y **F1** a nivel de tokens en un pequeño dataset manual.
4. Implementar una mini QA "open-domain" sobre un **corpus pequeño** de párrafos, usando un método sencillo de recuperación de contexto.
5. Analizar errores típicos y limitaciones del enfoque.


## 1) (Opcional) Instalación / actualización de librerías

In [1]:
# (Opcional) Instalación / actualización de librerías en Colab.
# Normalmente bastará con ejecutar esta celda una vez al principio del notebook.

# import sys
# pip = f"{sys.executable} -m pip"
# !{pip} install -q "transformers>=4.40.0"

## 2) Imports y configuración básica

In [2]:
from transformers import pipeline
import re
import numpy as np

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 3) Creación del pipeline de QA extractivo

In [3]:
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

# Prueba rápida
context = (
    "Natural language processing (NLP) is a subfield of artificial intelligence "
    "that focuses on the interaction between computers and humans through natural language."
)
question = "What does NLP focus on?"

result = qa_pipeline({"question": question, "context": context})
print(result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/question_answering.py:395: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(


{'score': 0.3301957845687866, 'start': 91, 'end': 160, 'answer': 'the interaction between computers and humans through natural language'}


## 4) Definición de ejemplos de contexto + preguntas

In [4]:
qa_examples = [
    {
        "context": (
            "Natural language processing (NLP) is a subfield of artificial intelligence "
            "that focuses on the interaction between computers and humans through natural language. "
            "Most NLP techniques rely on machine learning to derive meaning from human languages."
        ),
        "question": "What does NLP focus on?",
        "answer": "the interaction between computers and humans through natural language"
    },
    {
        "context": (
            "Machine learning is a method of data analysis that automates analytical model building. "
            "It is a branch of artificial intelligence based on the idea that systems can learn from data, "
            "identify patterns and make decisions with minimal human intervention."
        ),
        "question": "What is machine learning based on?",
        "answer": "the idea that systems can learn from data"
    },
    {
        "context": (
            "Neural networks are a set of algorithms, modeled loosely after the human brain, "
            "that are designed to recognize patterns. They interpret sensory data through a kind of "
            "machine perception, labeling or clustering of raw input."
        ),
        "question": "What are neural networks designed to recognize?",
        "answer": "patterns"
    },
    {
        "context": (
            "Deep learning is a subset of machine learning in artificial intelligence that has networks "
            "capable of learning unsupervised from data that is unstructured or unlabeled. "
            "Also known as deep neural learning or deep neural network."
        ),
        "question": "Deep learning is a subset of what?",
        "answer": "machine learning"
    },
]

for i, ex in enumerate(qa_examples):
    print(f"Ejemplo {i+1}")
    print("Context (primeras 30 palabras):", " ".join(ex["context"].split()[:30]), "...")
    print("Question:", ex["question"])
    print("Answer (gold):", ex["answer"])
    print()

Ejemplo 1
Context (primeras 30 palabras): Natural language processing (NLP) is a subfield of artificial intelligence that focuses on the interaction between computers and humans through natural language. Most NLP techniques rely on machine learning to ...
Question: What does NLP focus on?
Answer (gold): the interaction between computers and humans through natural language

Ejemplo 2
Context (primeras 30 palabras): Machine learning is a method of data analysis that automates analytical model building. It is a branch of artificial intelligence based on the idea that systems can learn from data, ...
Question: What is machine learning based on?
Answer (gold): the idea that systems can learn from data

Ejemplo 3
Context (primeras 30 palabras): Neural networks are a set of algorithms, modeled loosely after the human brain, that are designed to recognize patterns. They interpret sensory data through a kind of machine perception, labeling ...
Question: What are neural networks designed to reco

## 5) Función auxiliar `answer_question`

In [5]:
def answer_question(question: str, context: str):
    """Envuelve el pipeline de QA para una llamada simple."""
    return qa_pipeline({"question": question, "context": context})

# Prueba de la función
test_res = answer_question(qa_examples[0]["question"], qa_examples[0]["context"])
print(test_res)

{'score': 0.36132705211639404, 'start': 91, 'end': 160, 'answer': 'the interaction between computers and humans through natural language'}


## 6) Exploración de `answer`, `score` y spans

In [6]:
for i, ex in enumerate(qa_examples):
    pred = answer_question(ex["question"], ex["context"])
    print("=" * 80)
    print(f"Ejemplo {i+1}")
    print("Question:", ex["question"])
    print("Predicted answer:", pred["answer"])
    print("Score:", pred["score"])
    print("Gold answer:", ex["answer"])
    print()

Ejemplo 1
Question: What does NLP focus on?
Predicted answer: the interaction between computers and humans through natural language
Score: 0.36132705211639404
Gold answer: the interaction between computers and humans through natural language

Ejemplo 2
Question: What is machine learning based on?
Predicted answer: systems can learn from data, identify patterns and make decisions with minimal human intervention
Score: 0.12359260767698288
Gold answer: the idea that systems can learn from data

Ejemplo 3
Question: What are neural networks designed to recognize?
Predicted answer: patterns
Score: 0.9901171922683716
Gold answer: patterns

Ejemplo 4
Question: Deep learning is a subset of what?
Predicted answer: machine learning
Score: 0.5977093577384949
Gold answer: machine learning



## 7) Normalización de texto y métricas EM / F1

In [7]:
def normalize_text(s: str) -> str:
    """Normaliza el texto para comparar respuestas (minúsculas, sin puntuación extra)."""
    s = s.lower()
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def compute_exact_match(pred: str, gold: str) -> int:
    """Devuelve 1 si pred y gold coinciden exactamente tras normalizar, 0 si no."""
    return int(normalize_text(pred) == normalize_text(gold))

def compute_f1(pred: str, gold: str) -> float:
    """Calcula F1 a nivel de tokens entre pred y gold (tras normalizar)."""
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    pred_set = set(pred_tokens)
    gold_set = set(gold_tokens)
    common = pred_set & gold_set
    num_common = sum(min(pred_tokens.count(w), gold_tokens.count(w)) for w in common)

    if num_common == 0:
        return 0.0

    precision = num_common / len(pred_tokens)
    recall = num_common / len(gold_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return f1

# Pruebas rápidas
print("EM test:", compute_exact_match("Patterns.", "patterns"))
print("F1 test:", compute_f1("the interaction between computers and humans", "the interaction between computers and humans through natural language"))

EM test: 1
F1 test: 0.8


## 8) Evaluación en tu mini-dataset de QA

In [8]:
results = []

for i, ex in enumerate(qa_examples):
    pred = answer_question(ex["question"], ex["context"])
    pred_answer = pred["answer"]
    gold_answer = ex["answer"]

    em = compute_exact_match(pred_answer, gold_answer)
    f1 = compute_f1(pred_answer, gold_answer)

    results.append({"pred": pred_answer, "gold": gold_answer, "em": em, "f1": f1})

    print("=" * 80)
    print(f"Ejemplo {i+1}")
    print("Question:", ex["question"])
    print("Predicted:", pred_answer)
    print("Gold    :", gold_answer)
    print("EM:", em, "F1:", f1)
    print()

em_scores = [r["em"] for r in results]
f1_scores = [r["f1"] for r in results]

print("EM medio:", np.mean(em_scores))
print("F1 medio:", np.mean(f1_scores))

Ejemplo 1
Question: What does NLP focus on?
Predicted: the interaction between computers and humans through natural language
Gold    : the interaction between computers and humans through natural language
EM: 1 F1: 1.0

Ejemplo 2
Question: What is machine learning based on?
Predicted: systems can learn from data, identify patterns and make decisions with minimal human intervention
Gold    : the idea that systems can learn from data
EM: 0 F1: 0.45454545454545453

Ejemplo 3
Question: What are neural networks designed to recognize?
Predicted: patterns
Gold    : patterns
EM: 1 F1: 1.0

Ejemplo 4
Question: Deep learning is a subset of what?
Predicted: machine learning
Gold    : machine learning
EM: 1 F1: 1.0

EM medio: 0.75
F1 medio: 0.8636363636363636


## 9) Mini QA "open-domain" con recuperación sencilla

In [9]:
corpus = [
    (
        "Python is a popular programming language created by Guido van Rossum and first released in 1991. "
        "It is widely used in web development, data analysis, artificial intelligence, and scientific computing."
    ),
    (
        "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
        "It is named after the engineer Gustave Eiffel, whose company designed and built the tower."
    ),
    (
        "The Pacific Ocean is the largest and deepest of Earth's oceanic divisions. "
        "It extends from the Arctic Ocean in the north to the Southern Ocean in the south and is bounded by Asia and Australia in the west and the Americas in the east."
    ),
    (
        "The Mona Lisa is a half-length portrait painting by Italian artist Leonardo da Vinci. "
        "It is considered an archetypal masterpiece of the Italian Renaissance and has been described as the best-known, the most visited, and the most written about work of art in the world."
    ),
]

open_questions = [
    "Who created the Python programming language?",
    "Where is the Eiffel Tower located?",
    "What is the largest ocean on Earth?",
    "Who painted the Mona Lisa?",
]

open_answers_gold = [
    "Guido van Rossum",
    "on the Champ de Mars in Paris, France",  # o variaciones equivalentes
    "the Pacific Ocean",
    "Leonardo da Vinci",
]

print("Número de documentos en el corpus:", len(corpus))
print("Número de preguntas open-domain:", len(open_questions))

Número de documentos en el corpus: 4
Número de preguntas open-domain: 4


## 10) Recuperación de contexto: escoger el mejor párrafo

In [10]:
def tokenize_simple(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    return text.split()

def score_overlap(question: str, doc: str) -> int:
    """Devuelve el número de palabras en común entre pregunta y documento (muy simple)."""
    q_tokens = set(tokenize_simple(question))
    d_tokens = set(tokenize_simple(doc))
    return len(q_tokens & d_tokens)

def retrieve_best_context(question: str, corpus: list):
    """Selecciona el documento del corpus con mayor solapamiento léxico con la pregunta."""
    best_score = -1
    best_doc = None
    best_idx = -1
    for i, doc in enumerate(corpus):
        s = score_overlap(question, doc)
        if s > best_score:
            best_score = s
            best_doc = doc
            best_idx = i
    return best_doc, best_idx

# Prueba rápida de recuperación
for q in open_questions:
    best_doc, idx = retrieve_best_context(q, corpus)
    print("=" * 40)
    print("Question:", q)
    print("Selected doc idx:", idx)
    print("Doc (primeras 20 palabras):", " ".join(best_doc.split()[:20]), "...")

Question: Who created the Python programming language?
Selected doc idx: 0
Doc (primeras 20 palabras): Python is a popular programming language created by Guido van Rossum and first released in 1991. It is widely used ...
Question: Where is the Eiffel Tower located?
Selected doc idx: 1
Doc (primeras 20 palabras): The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after ...
Question: What is the largest ocean on Earth?
Selected doc idx: 2
Doc (primeras 20 palabras): The Pacific Ocean is the largest and deepest of Earth's oceanic divisions. It extends from the Arctic Ocean in the ...
Question: Who painted the Mona Lisa?
Selected doc idx: 3
Doc (primeras 20 palabras): The Mona Lisa is a half-length portrait painting by Italian artist Leonardo da Vinci. It is considered an archetypal masterpiece ...


## 11) QA "open-domain" sobre tu mini-corpus

In [11]:
open_results = []

for q, gold in zip(open_questions, open_answers_gold):
    best_ctx, idx = retrieve_best_context(q, corpus)
    pred = answer_question(q, best_ctx)
    pred_answer = pred["answer"]

    em = compute_exact_match(pred_answer, gold)
    f1 = compute_f1(pred_answer, gold)

    open_results.append({"question": q, "pred": pred_answer, "gold": gold, "em": em, "f1": f1})

    print("=" * 80)
    print("Question:", q)
    print("Context idx:", idx)
    print("Context (primeras 25 palabras):", " ".join(best_ctx.split()[:25]), "...")
    print("Predicted answer:", pred_answer)
    print("Gold answer:", gold)
    print("EM:", em, "F1:", f1)
    print()

em_open = np.mean([r["em"] for r in open_results])
f1_open = np.mean([r["f1"] for r in open_results])
print("Open-domain EM medio:", em_open)
print("Open-domain F1 medio:", f1_open)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Question: Who created the Python programming language?
Context idx: 0
Context (primeras 25 palabras): Python is a popular programming language created by Guido van Rossum and first released in 1991. It is widely used in web development, data analysis, ...
Predicted answer: Guido van Rossum
Gold answer: Guido van Rossum
EM: 1 F1: 1.0

Question: Where is the Eiffel Tower located?
Context idx: 1
Context (primeras 25 palabras): The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose ...
Predicted answer: Champ de Mars
Gold answer: on the Champ de Mars in Paris, France
EM: 0 F1: 0.5454545454545454

Question: What is the largest ocean on Earth?
Context idx: 2
Context (primeras 25 palabras): The Pacific Ocean is the largest and deepest of Earth's oceanic divisions. It extends from the Arctic Ocean in the north to the Southern Ocean ...
Predicted answer: The Pacific Ocean
Gold answer: the Pacific Ocean
EM: 1 F

## 12) Conclusiones

En este notebook hemos visto, de manera práctica:

- Cómo usar un modelo **extractivo** de QA basado en Transformers (RoBERTa finetuneado en SQuAD).
- Cómo evaluar el modelo en un mini-dataset manual con métricas tipo SQuAD (EM y F1).
- Cómo construir un sistema de QA "open-domain" muy sencillo combinando:
  1. Una fase de **recuperación** basada en solapamiento léxico.
  2. El modelo de QA extractivo sobre el contexto recuperado.
- Cómo la calidad de la recuperación impacta directamente en el rendimiento del sistema global.
- Qué tipos de errores aparecen: respuestas razonables pero no exactas, spans demasiado largos/cortos, respuestas al contexto equivocado, etc.

Estas observaciones se conectan con los apartados **4.9–4.11** del Tema 4 (QA), donde se discuten:
- QA extractivo vs QA generativo.
- Modelos basados en Transformers para SQuAD y datasets similares.
- Arquitecturas tipo **retrieval + reader** para QA open-domain.

Como extensiones posibles, podrías:
- Reemplazar la recuperación léxica por una recuperación basada en embeddings (p.ej. usando modelos tipo sentence-transformers).
- Probar un modelo de QA distinto (BERT, DistilBERT, modelos multilingües) y comparar resultados.
- Ampliar el corpus y el conjunto de preguntas para obtener estadísticas más fiables.
